In [1]:
import sys
from PIL import Image
import os
import numpy as np
import openslide
from tqdm import tqdm
import pickle

In [2]:
sys.path.append(os.path.join(os.getcwd(), 'histocartography'))
from histocartography.preprocessing import NucleiExtractor, DeepFeatureExtractor, KNNGraphBuilder
from histocartography.visualization import OverlayGraphVisualization, InstanceImageVisualization

/data/mn27889/miniconda3/envs/pbt-histo/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initiating all the required modules

In [3]:
nuclei_detector = NucleiExtractor()
feature_extractor = DeepFeatureExtractor(architecture='resnet34', patch_size=224, resize_size=224)
knn_graph_builder = KNNGraphBuilder(k=5, thresh=50, add_loc_feats=True)

File already downloaded.


/data/mn27889/pbt-histocartography/histocartography/histocartography/preprocessing/nuclei_extraction.py:88: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model = torch.

Processing files one by one

In [5]:
# raw_wsi_dir = "/data/jy26539/data/wsi_raw"
raw_wsi_dir = "/data/mn27889/pbt-histocartography/tcga_download/tcga_gbm_lgg_pedi"
wsi_graph_dir = "wsi_graphs_tcga"
os.makedirs(wsi_graph_dir, exist_ok=True)
wsi_names = os.listdir(raw_wsi_dir)
print(f"Found {len(wsi_names)} WSIs in {raw_wsi_dir}.")

Found 283 WSIs in /data/mn27889/pbt-histocartography/tcga_download/tcga_gbm_lgg_pedi.


In [ ]:
wsi_resolution_level = 1 # Change this to adjust the resolution level for processing (0 is highest resolution)
features_dict = {}
cell_graph_dict = {}

for i in tqdm(range(len(wsi_names)), desc="Processing"):
    wsi_name = wsi_names[i]
    print(f"Processing {wsi_name} ({i+1}/{len(wsi_names)})...")
    wsi_path = os.path.join(raw_wsi_dir, wsi_name)
    wsi_graph_path = os.path.join(wsi_graph_dir, f"{wsi_name}_Graph.png")
    try:
        slide = openslide.OpenSlide(wsi_path)
        svs_image = slide.read_region((0, 0), wsi_resolution_level, slide.level_dimensions[wsi_resolution_level]).convert('RGB')
        svs_image_np = np.array(svs_image)
        
        nuclei_map, nuclei_centers = nuclei_detector.process(svs_image_np)
        features = feature_extractor.process(svs_image_np, nuclei_map)
        features_dict[wsi_name] = features.detach().cpu().numpy()  # Store features for later use
        
        if len(nuclei_centers) > 5:
            cell_graph = knn_graph_builder.process(nuclei_map, features)
            cell_graph_dict[wsi_name] = cell_graph  # Store cell graph for later use
            
            visualizer = OverlayGraphVisualization(instance_visualizer=InstanceImageVisualization(instance_style="filled+outline"))
            viz_cg = visualizer.process(canvas=svs_image_np, graph=cell_graph, instance_map=nuclei_map)
            viz_cg.save(wsi_graph_path)
            print(f"Processed {wsi_name} and saved graph visualization to {wsi_graph_path}.")
        else:
            print(f"Less than 5 nuclei detected in {wsi_name}. Skipping graph construction and visualization.")

        slide.close()
    except Exception as e:
        print(f"!!!!!!!!!!!Error processing {wsi_name} ({i+1}/{len(wsi_names)}): {e}!!!!!!!!!")

Save the feature_dict and cell_graph_dict as pickle file

In [ ]:
with open('wsi_raw_graph_features_tcga.pkl', 'wb') as f:
    pickle.dump(features_dict, f)

with open('wsi_raw_cell_graphs_tcga.pkl', 'wb') as f:
    pickle.dump(cell_graph_dict, f)

Read the feature_dict and cell_graph_dict

In [ ]:
with open('wsi_raw_graph_features_tcga.pkl', 'rb') as f:
    features_dict = pickle.load(f)

with open('wsi_raw_cell_graphs_tcga.pkl', 'rb') as f:
    cell_graph_dict = pickle.load(f)